<div style="background:linear-gradient(135deg,#0f0c29 0%,#302b63 55%,#24243e 100%);border-radius:18px;padding:32px 24px 24px;margin:8px 0;box-shadow:0 10px 40px rgba(0,0,0,0.5);text-align:center;">

  <div style="font-size:2.8em;margin:0 0 6px;text-align:center">🎨</div>
  <h1 style="color:#fff;margin:0 0 6px;font-size:2.3em;font-weight:700;letter-spacing:-0.5px;text-align:center">Krea-2-Turbo</h1>
  <p style="color:#c0c0ff;margin:0 0 4px;font-size:1.1em;font-weight:300;text-align:center">Google Colab Edition</p>
  <p style="color:#7a7a9a;margin:0 0 20px;font-size:0.88em;text-align:center">Fast Text-to-Image | 12B DiT | INT8 Quantized | Wan2GP Engine</p>

  <div style="margin-bottom:18px;text-align:center">
    <a href="https://www.youtube.com/@thebuildai?sub_confirmation=1"><img src="https://img.shields.io/badge/YouTube-SUBSCRIBE-FF0000?style=for-the-badge&amp;logo=youtube&amp;logoColor=white" alt="YouTube Subscribe"></a>
    <a href="https://www.instagram.com/thebuildai/"><img src="https://img.shields.io/badge/Instagram-FOLLOW-E4405F?style=for-the-badge&amp;logo=instagram&amp;logoColor=white" alt="Instagram Follow"></a>
    <a href="https://www.tiktok.com/@the.build.ai"><img src="https://img.shields.io/badge/TikTok-FOLLOW-000000?style=for-the-badge&amp;logo=tiktok&amp;logoColor=white" alt="TikTok Follow"></a>
    <a href="https://github.com/cafermutluozkan/free-ai-notebooks"><img src="https://img.shields.io/badge/GitHub-STAR_%E2%98%85-181717?style=for-the-badge&amp;logo=github&amp;logoColor=white" alt="GitHub Star"></a>
  </div>

  <div style="display:inline-block;background:rgba(255,200,0,0.10);border:1px solid rgba(255,215,0,0.35);border-radius:10px;padding:10px 24px;text-align:center">
    <span style="color:#ffd700;font-size:0.95em">🔔 Have you <strong>subscribed</strong> to the channel?&nbsp;&nbsp;•&nbsp;&nbsp;⭐ Don't forget to <strong>star us on GitHub!</strong></span>
  </div>

</div>

---

### 🚀 Quick Start — Google Colab
1. **Runtime → Change runtime type → T4 GPU**
2. Run **Cell 1** → **Cell 2** → **Cell 3** in order
3. Open the **Gradio public URL** printed in Cell 3

> 💡 Total disk usage ~18 GB. Make sure Colab session has enough disk.

In [ ]:
# @title ⚙️ Cell 1 — Install & Setup
import os, gc, subprocess, sys

gc.collect()
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,garbage_collection_threshold:0.6'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

def _version_tuple(v):
    parts = []
    for p in v.split('.'):
        num = ''
        for ch in p:
            if ch.isdigit():
                num += ch
            else:
                break
        parts.append(int(num) if num else 0)
    return tuple(parts)

def _version_ok(pkg, min_version):
    try:
        import importlib.metadata as md
        installed = md.version(pkg)
        return _version_tuple(installed) >= _version_tuple(min_version)
    except Exception:
        return False

_REQUIRED = {'numpy': '2.2', 'scipy': '1.14', 'transformers': '4.46', 'accelerate': '0.34'}
_needs_restart = any(not _version_ok(p, v) for p, v in _REQUIRED.items())

if _needs_restart:
    print('Upgrading numpy/scipy/transformers/accelerate (one-time)...')
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--no-cache-dir',
         'numpy>=2.2', 'scipy>=1.14', 'transformers>=4.46', 'accelerate>=0.34'],
        check=True)
    print('Done. Restarting the kernel automatically to load the new versions...')
    print('>>> After the kernel restarts, just RUN THIS CELL AGAIN. <<<')
    import os as _os
    _os.kill(_os.getpid(), 9)
else:
    print('numpy/scipy/transformers/accelerate versions OK - continuing setup.')

import torch
if torch.cuda.is_available():
    print(f'GPU  : {torch.cuda.get_device_name(0)}')
    print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')
else:
    print('No GPU - go to Runtime -> Change runtime type -> T4 GPU')

# Clone Wan2GP
print('[1/3] Setting up Wan2GP...')
if not os.path.exists('Wan2GP'):
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/DeepBeepMeep/Wan2GP.git'], check=True)
    print('  Cloned Wan2GP')
else:
    print('  Wan2GP already present')

# Install dependencies
print('[2/3] Installing dependencies...')
r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     '-r', 'Wan2GP/requirements.txt'],
    check=False)
if r.returncode != 0:
    print('  Some requirements failed - continuing anyway')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'mmgp', 'gradio'],
    check=True)

# Patch krea2_main.py for float16 / T4 compatibility
print('[3/3] Patching krea2_main.py...')
_kp = os.path.join('Wan2GP', 'models', 'krea2', 'krea2_main.py')
if os.path.exists(_kp):
    with open(_kp, 'r') as _f:
        _src = _f.read().replace('\r\n', '\n')
    _ch = False
    if 'dtype = torch.bfloat16' in _src:
        _src = _src.replace('dtype = torch.bfloat16',
                            '# dtype = torch.bfloat16  # patched: float16 for T4')
        _ch = True
    _old = ('offload.load_model_data(transformer, model_filename, '
            'writable_tensors=False, preprocess_sd=preprocess_sd, default_dtype=dtype)')
    _new = ('def tf_preprocess(sd):\n'
            '        sd = preprocess_sd(sd)\n'
            '        return {k: v.to(dtype) if (isinstance(v, torch.Tensor) and v.is_floating_point()) else v for k, v in sd.items()}\n'
            '    offload.load_model_data(transformer, model_filename, '
            'writable_tensors=False, preprocess_sd=tf_preprocess, default_dtype=dtype)')
    if _old in _src:
        _src = _src.replace(_old, _new); _ch = True
    _oldv = 'offload.load_model_data(vae, filename, writable_tensors=False, default_dtype=dtype)'
    _newv = ('def vae_preprocess(sd):\n'
             '        return {k: v.to(dtype) if (isinstance(v, torch.Tensor) and v.is_floating_point()) else v for k, v in sd.items()}\n'
             '    offload.load_model_data(vae, filename, writable_tensors=False, preprocess_sd=vae_preprocess, default_dtype=dtype)')
    if _oldv in _src:
        _src = _src.replace(_oldv, _newv); _ch = True
    if _ch:
        with open(_kp, 'w') as _f: _f.write(_src)
        print('  krea2_main.py patched for float16')
    else:
        print('  krea2_main.py already patched')
else:
    print('  krea2_main.py not found yet')

print('\nSetup complete - run Cell 2 to download models.')


In [ ]:
# @title 📥 Cell 2 — Download Models
import os
from pathlib import Path
from huggingface_hub import hf_hub_download

REPO          = "DeepBeepMeep/krea-2"
QWEN_IMG_REPO = "DeepBeepMeep/Qwen_image"
MODEL_DIR     = "Wan2GP/models"
Path(MODEL_DIR).mkdir(parents=True, exist_ok=True)

def dl(repo, filename, dest_dir, dest_name=None):
    Path(dest_dir).mkdir(parents=True, exist_ok=True)
    name = dest_name or os.path.basename(filename)
    fp   = os.path.join(dest_dir, name)
    if os.path.exists(fp):
        print(f"  ✓ {name} already downloaded"); return
    print(f"  ↓ Downloading {name} ...")
    hf_hub_download(repo_id=repo, filename=filename,
                    local_dir=dest_dir, local_dir_use_symlinks=False)
    tmp = os.path.join(dest_dir, filename)
    if os.path.exists(tmp) and tmp != fp:
        os.makedirs(os.path.dirname(fp), exist_ok=True)
        os.rename(tmp, fp)
    print(f"  ✅ {name}")

# Transformer
dl(REPO, "Krea2Turbo_quanto_bf16_int8.safetensors", MODEL_DIR)

# Text encoder
TE = "Qwen3-VL-4B-Instruct"
for f in ["Qwen3-VL-4B-Instruct_quanto_bf16_int8.safetensors",
          "config.json", "tokenizer.json", "tokenizer_config.json", "chat_template.jinja"]:
    dl(REPO, f"{TE}/{f}", os.path.join(MODEL_DIR, TE), dest_name=f)

# VAE
for f in ["qwen_vae.safetensors", "qwen_vae_config.json"]:
    dl(QWEN_IMG_REPO, f, MODEL_DIR)

print("\n✅ All models ready — run Cell 3 to launch Gradio.")


In [ ]:
# @title 🚀 Cell 3 — Load Model & Launch Gradio
import os, sys, gc, time, random, traceback, zipfile, tempfile
import numpy as np
from PIL import Image

WAN2GP    = os.path.abspath("Wan2GP")
MODEL_DIR = os.path.join(WAN2GP, "models")

os.chdir(WAN2GP)
if WAN2GP not in sys.path:
    sys.path.insert(0, WAN2GP)

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,garbage_collection_threshold:0.6"
os.environ["TOKENIZERS_PARALLELISM"]  = "false"

import torch
import gradio as gr
from mmgp import offload
from shared.utils import files_locator as fl
from models.krea2.krea2_handler import family_handler

fl.set_checkpoints_paths(["models", "ckpts", "."])

# ── Verify paths ──────────────────────────────────────────────────────────────
TRANSFORMER   = os.path.join(MODEL_DIR, "Krea2Turbo_quanto_bf16_int8.safetensors")
TEXT_ENCODER  = os.path.join(MODEL_DIR, "Qwen3-VL-4B-Instruct",
                             "Qwen3-VL-4B-Instruct_quanto_bf16_int8.safetensors")
if not os.path.exists(TRANSFORMER):
    raise FileNotFoundError(f"Transformer not found: {TRANSFORMER}\nRun Cell 2 first.")
if not os.path.exists(TEXT_ENCODER):
    raise FileNotFoundError(f"Text encoder not found: {TEXT_ENCODER}\nRun Cell 2 first.")

print(f"Transformer  : {os.path.basename(TRANSFORMER)}")
print(f"Text encoder : {os.path.basename(TEXT_ENCODER)}")

# ── Load ──────────────────────────────────────────────────────────────────────
print("\nLoading Krea-2-Turbo (INT8) — this takes 1–3 min...")
base_model_type = "krea2_turbo"
model_def = family_handler.query_model_def(base_model_type, {})

krea_model, pipe = family_handler.load_model(
    model_filename=TRANSFORMER,
    model_type="krea2_turbo",
    base_model_type=base_model_type,
    model_def=model_def,
    quantizeTransformer=False,
    dtype=torch.float16,
    VAE_dtype=torch.float16,
    text_encoder_filename=TEXT_ENCODER,
)

offload.profile(
    pipe, profile_no=2, quantizeTransformer=False,
    convertWeightsFloatTo=torch.float16,
    pinnedMemory=True, asyncTransfers=True,
    budgets={"transformer": 13000, "text_encoder": 4500, "vae": 2000, "*": 1000},
)
offload.shared_state["_attention"] = "sdpa"
print("✅ Model ready!")

# ── Prompt enhancer ───────────────────────────────────────────────────────────
def _enhance(prompt, style_preset="None"):
    if not prompt:
        return ""
    p = prompt.lower()
    if any(k in p for k in ["photorealistic", "hyperrealistic", "highly detailed", "8k resolution"]):
        enhanced = prompt
    elif any(w in p for w in ["person","woman","man","girl","boy","portrait","face"]):
        enhanced = prompt + ", 85mm lens, f/1.8, natural skin texture, professional studio lighting, highly detailed, 8k"
    elif any(w in p for w in ["cyberpunk","futuristic","sci-fi","robot","neon"]):
        enhanced = prompt + ", cyberpunk aesthetic, neon glow, volumetric fog, cinematic lighting, ray tracing, 8k"
    elif any(w in p for w in ["forest","mountain","ocean","landscape","lake","river","sky","sunset"]):
        enhanced = prompt + ", golden hour, national geographic style, volumetric mist, wide-angle lens, 8k"
    elif any(w in p for w in ["dragon","magic","wizard","elf","castle","fantasy"]):
        enhanced = prompt + ", glowing magical particles, ethereal light, highly detailed fantasy art, 8k"
    elif any(w in p for w in ["anime","illustration","drawing","painting","digital art"]):
        enhanced = prompt + ", vibrant color palette, clean line art, beautiful lighting, anime key visual, masterpiece"
    else:
        enhanced = prompt + ", highly detailed, photorealistic, cinematic lighting, masterpiece, 8k"
    styles = {"Cinematic": "cinematic lighting, dramatic shadows, film grain, masterpiece",
               "Photographic": "professional photography, 35mm lens, f/2.8, depth of field, photorealistic",
               "Anime": "anime key visual, vibrant color palette, clean line art, highly detailed",
               "Cyberpunk": "cyberpunk aesthetic, neon lights, volumetric smoke, high contrast",
               "Fantasy": "mythical atmosphere, glowing particles, ethereal light, digital painting"}
    if style_preset and style_preset != "None" and style_preset in styles:
        enhanced += ", " + styles[style_preset]
    return enhanced

# ── Resolution helper ─────────────────────────────────────────────────────────
def _dims(aspect, base):
    b = {"1024px (Standard)": 1024, "1536px (High)": 1536, "2048px (2K Ultra)": 2048}.get(base, 1024)
    if aspect == "1:1 Square":        return b, b
    elif aspect == "16:9 Landscape":  return b, int((b*9/16)//16*16)
    elif aspect == "9:16 Portrait":   return int((b*9/16)//16*16), b
    elif aspect == "4:3 Standard":    return b, int((b*3/4)//16*16)
    elif aspect == "3:4 Portrait":    return int((b*3/4)//16*16), b
    return b, b

# ── Generate ──────────────────────────────────────────────────────────────────
@torch.inference_mode()
def generate(prompt, negative_prompt, steps, aspect_ratio, resolution,
             seed, num_images, progress=gr.Progress(track_tqdm=True)):
    try:
        gc.collect(); torch.cuda.empty_cache()
        yield gr.update(value=None, selected_index=None), "⏳ Starting...", gr.update(visible=False)

        width, height = _dims(aspect_ratio, resolution)
        init_seed = random.randint(0, 2**32-1) if (seed is None or seed < 0) else int(seed)
        images, seeds_used = [], []
        t0 = time.time()

        for i in range(int(num_images)):
            s = (init_seed + i) % (2**32)
            seeds_used.append(s)

            def cb(step_idx, latent, is_start, override_num_inference_steps=None, **kw):
                pct = (i + (step_idx+1)/int(steps)) / int(num_images)
                progress(min(pct, 0.99),
                         desc=f"Image {i+1}/{num_images} — step {step_idx+1}/{steps}")

            result = krea_model.generate(
                seed=s, input_prompt=prompt,
                n_prompt=negative_prompt.strip() or None,
                sampling_steps=int(steps),
                width=width, height=height,
                guide_scale=0.0, batch_size=1,
                callback=cb,
                loras_slists={"phase1": []},
            )
            if result is None:
                raise RuntimeError(f"Image {i+1} generation returned None")

            img = Image.fromarray(result[:, 0].permute(1, 2, 0).numpy())
            images.append(img)
            yield gr.update(value=images, selected_index=0),                   f"⏳ Generated {len(images)}/{num_images}...", gr.update(visible=False)

        # ZIP for multiple images
        zip_path = None
        if len(images) > 1:
            tmp = tempfile.NamedTemporaryFile(suffix=".zip", delete=False)
            zip_path = tmp.name
            with zipfile.ZipFile(zip_path, "w") as zf:
                for idx, img in enumerate(images):
                    p = tempfile.mktemp(suffix=".png")
                    img.save(p)
                    zf.write(p, f"krea2_{idx+1}_seed_{seeds_used[idx]}.png")
                    try: os.remove(p)
                    except: pass

        elapsed = time.time() - t0
        status = (f"✅ {len(images)} image(s) in {elapsed:.1f}s | "
                  f"Seeds: {seeds_used} | {width}x{height}")
        yield gr.update(value=images, selected_index=0), status,               gr.update(value=zip_path, visible=(zip_path is not None))

    except Exception as e:
        traceback.print_exc()
        gc.collect(); torch.cuda.empty_cache()
        yield gr.update(value=None, selected_index=None), f"❌ Error: {e}", gr.update(visible=False)

# ── Gradio UI ─────────────────────────────────────────────────────────────────
with gr.Blocks(theme=gr.themes.Soft(), title="Krea-2-Turbo | TheBuildAI") as demo:
    gr.Markdown(
        "# 🎨 Krea-2-Turbo — Fast Text-to-Image\n"
        "**TheBuildAI** | Wan2GP + mmgp | INT8 Quantized | 12B DiT"
    )
    with gr.Row():
        with gr.Column(scale=3):
            prompt = gr.Textbox(label="📝 Prompt", lines=3,
                placeholder="A small dark cat walking down an abandoned cobblestone street at dusk...")
            with gr.Row():
                enhance_btn = gr.Button("✨ Enhance Prompt", variant="secondary")
                style_preset = gr.Dropdown(
                    ["None","Cinematic","Photographic","Anime","Cyberpunk","Fantasy"],
                    value="None", label="🎨 Style Preset")
            with gr.Row():
                aspect_ratio = gr.Dropdown(
                    ["16:9 Landscape","9:16 Portrait","1:1 Square","4:3 Standard","3:4 Portrait"],
                    value="16:9 Landscape", label="📐 Aspect Ratio")
                resolution = gr.Dropdown(
                    ["1024px (Standard)","1536px (High)","2048px (2K Ultra)"],
                    value="1024px (Standard)", label="🖥️ Resolution")
            with gr.Row():
                steps     = gr.Slider(1, 12, 8, step=1, label="⚡ Steps")
                num_images = gr.Slider(1, 4, 1, step=1, label="🖼️ Images")
            with gr.Accordion("⚙️ Advanced", open=False):
                negative_prompt = gr.Textbox(label="🚫 Negative Prompt",
                    placeholder="low quality, blurry, distorted, bad anatomy", lines=2)
                seed = gr.Number(-1, label="🎲 Seed (-1 = random)", precision=0)
            with gr.Row():
                gen_btn   = gr.Button("🎨 Generate", variant="primary", size="lg")
                clear_btn = gr.Button("🗑️ Clear",    variant="secondary", size="lg")

        with gr.Column(scale=1):
            gallery   = gr.Gallery(label="🖼️ Output", columns=2, rows=2,
                                   object_fit="contain", preview=True)
            zip_out   = gr.File(label="📦 Download ZIP", visible=False)
            status_out = gr.Textbox(label="ℹ️ Status", interactive=False)

    enhance_btn.click(fn=lambda p, s: _enhance(p, s),
                      inputs=[prompt, style_preset], outputs=[prompt])
    gen_btn.click(fn=generate,
                  inputs=[prompt, negative_prompt, steps, aspect_ratio,
                          resolution, seed, num_images],
                  outputs=[gallery, status_out, zip_out])
    clear_btn.click(
        fn=lambda: ("", "", 8, "16:9 Landscape", "1024px (Standard)",
                    -1, 1, "None", gr.update(value=None, selected_index=None), "", gr.update(visible=False)),
        outputs=[prompt, negative_prompt, steps, aspect_ratio, resolution,
                 seed, num_images, style_preset, gallery, status_out, zip_out])

    gr.Markdown(
        "---\n"
        "> 🔔 Enjoying this? "
        "[Subscribe on YouTube](https://www.youtube.com/@thebuildai?sub_confirmation=1) "
        "and [Star on GitHub](https://github.com/cafermutluozkan/free-ai-notebooks)!"
    )

demo.queue()
demo.launch(share=True)
